# 01 — Explore MedMNIST data

Exploratory analysis of the MedMNIST datasets relevant to this project:

- **PneumoniaMNIST** — our planned in-distribution (ID) dataset; binary chest X-ray task.
- **BloodMNIST** — multi-class blood-cell microscopy (RGB); candidate **far-OOD** dataset.
- **OrganAMNIST** — multi-class CT abdominal slices (grayscale); candidate **near-OOD** dataset.

Goals:
1. Understand sizes, class balance, and what the images look like.
2. Produce publication-quality figures for the 06.05.26 presentation.

**No training, no preprocessing logic that belongs in `src/` — exploration only.**

## 1. Setup

In [6]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from medmnist import INFO, BloodMNIST, OrganAMNIST, PneumoniaMNIST

# Reproducibility for any sampling we do below.
RNG = np.random.default_rng(0)

# Where to download MedMNIST .npz files (gitignored via data/).
DATA_ROOT = Path("../data").resolve()
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Where to save figures for the slides.
FIG_DIR = Path("figures").resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Clean publication style.
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# Consistent dataset color palette across the whole notebook.
DATASET_COLORS = {
    "PneumoniaMNIST": "#1f77b4",  # blue
    "BloodMNIST":     "#ff7f0e",  # orange
    "OrganAMNIST":    "#2ca02c",  # green
}

print("Data root:", DATA_ROOT)
print("Figures:  ", FIG_DIR)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Load all three datasets (all splits) at 28x28. download=True is idempotent.
def load_splits(cls):
    return {
        split: cls(split=split, download=True, root=str(DATA_ROOT), size=28)
        for split in ("train", "val", "test")
    }

datasets = {
    "PneumoniaMNIST": load_splits(PneumoniaMNIST),
    "BloodMNIST":     load_splits(BloodMNIST),
    "OrganAMNIST":    load_splits(OrganAMNIST),
}

# medmnist .npz keys we'll reuse a lot.
def imgs(ds):  return ds.imgs    # uint8 array, shape (N, H, W) or (N, H, W, C)
def labels(ds): return ds.labels.squeeze()  # shape (N,) for single-label tasks

## 2. Dataset overview

Sizes, classes, image shape, task type.

In [ ]:
def dataset_row(name, splits):
    info = INFO[splits["train"].flag]
    train = splits["train"]
    img = imgs(train)
    h, w = img.shape[1], img.shape[2]
    channels = 1 if img.ndim == 3 else img.shape[3]
    class_names = list(info["label"].values())
    return {
        "dataset": name,
        "task": info["task"],
        "n_train": len(train),
        "n_val":   len(splits["val"]),
        "n_test":  len(splits["test"]),
        "n_classes": len(class_names),
        "image_shape": f"{h}x{w}x{channels}",
        "class_names": ", ".join(class_names),
    }

overview = pd.DataFrame([dataset_row(n, s) for n, s in datasets.items()])
overview

## 3. Class distribution

Per-dataset class counts in the **train** split, with a special look at PneumoniaMNIST's imbalance.

In [ ]:
def class_counts(ds):
    info = INFO[ds.flag]
    names = list(info["label"].values())
    y = labels(ds)
    counts = np.bincount(y, minlength=len(names))
    return names, counts

def plot_class_distribution(name, ds, color, ax=None, save=True):
    names, counts = class_counts(ds)
    total = counts.sum()
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(max(4, 0.6 * len(names) + 2), 4))
    bars = ax.bar(range(len(names)), counts, color=color, edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_ylabel("# train samples")
    ax.set_title(f"{name} — train class distribution (n={total})")
    ymax = counts.max()
    ax.set_ylim(0, ymax * 1.18)
    for bar, c in zip(bars, counts):
        pct = 100 * c / total
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{c}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=9)
    if standalone and save:
        out = FIG_DIR / f"class_distribution_{name.lower()}.png"
        fig.savefig(out)
        print("saved", out.name)
    return ax

for name, splits in datasets.items():
    plot_class_distribution(name, splits["train"], DATASET_COLORS[name])
    plt.show()

In [ ]:
# Side-by-side comparison figure for the slides.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, splits) in zip(axes, datasets.items()):
    plot_class_distribution(name, splits["train"], DATASET_COLORS[name], ax=ax, save=False)
fig.suptitle("Train class distributions across the three MedMNIST datasets", y=1.02, fontsize=13)
fig.tight_layout()
out = FIG_DIR / "class_distribution_all.png"
fig.savefig(out)
print("saved", out.name)
plt.show()

## 4. Sample images

What do the images actually look like? For each dataset we pick a handful of classes and show 5 random examples per class.

In [ ]:
def sample_grid(name, ds, classes_to_show=None, n_per_class=5, save=True):
    info = INFO[ds.flag]
    names = list(info["label"].values())
    y = labels(ds)
    X = imgs(ds)
    if classes_to_show is None:
        classes_to_show = list(range(len(names)))

    n_rows, n_cols = len(classes_to_show), n_per_class
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 1.4, n_rows * 1.55))
    if n_rows == 1:
        axes = axes[None, :]

    is_rgb = (X.ndim == 4)
    for r, cls in enumerate(classes_to_show):
        idxs = np.where(y == cls)[0]
        pick = RNG.choice(idxs, size=min(n_per_class, len(idxs)), replace=False)
        for c in range(n_cols):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
            if c < len(pick):
                im = X[pick[c]]
                ax.imshow(im if is_rgb else im, cmap=None if is_rgb else "gray")
            if c == 0:
                ax.set_ylabel(names[cls], rotation=0, ha="right", va="center",
                              fontsize=10, labelpad=8)
    fig.suptitle(f"{name} — sample images", fontsize=12)
    fig.tight_layout()
    if save:
        out = FIG_DIR / f"samples_{name.lower()}.png"
        fig.savefig(out)
        print("saved", out.name)
    return fig

# PneumoniaMNIST: both classes.
sample_grid("PneumoniaMNIST", datasets["PneumoniaMNIST"]["train"])
plt.show()

In [ ]:
# BloodMNIST: pick the first 6 classes for a readable grid.
sample_grid("BloodMNIST", datasets["BloodMNIST"]["train"],
            classes_to_show=list(range(6)))
plt.show()

In [ ]:
# OrganAMNIST: pick 6 evenly-spaced classes across the 11.
n_organ_classes = len(INFO[datasets["OrganAMNIST"]["train"].flag]["label"])
picked = np.linspace(0, n_organ_classes - 1, 6).astype(int).tolist()
sample_grid("OrganAMNIST", datasets["OrganAMNIST"]["train"],
            classes_to_show=picked)
plt.show()

## 5. Image statistics

Pixel intensity distributions, mean image per Pneumonia class, and per-channel stats for the RGB dataset.

In [ ]:
# Overlay pixel intensity histograms across the three datasets.
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, splits in datasets.items():
    X = imgs(splits["train"]).astype(np.float32)
    # Subsample to keep histogram cheap and visually comparable.
    flat = X.reshape(-1)
    if flat.size > 2_000_000:
        flat = RNG.choice(flat, size=2_000_000, replace=False)
    ax.hist(flat, bins=64, range=(0, 255), density=True, alpha=0.45,
            color=DATASET_COLORS[name], label=name)
ax.set_xlabel("pixel intensity (0–255)")
ax.set_ylabel("density")
ax.set_title("Train pixel intensity distributions (overlaid)")
ax.legend()
out = FIG_DIR / "pixel_intensity_overlay.png"
fig.savefig(out)
print("saved", out.name)
plt.show()

In [ ]:
# Mean image per PneumoniaMNIST class — sanity check for obvious bias / artifacts.
pneu = datasets["PneumoniaMNIST"]["train"]
names = list(INFO[pneu.flag]["label"].values())
X, y = imgs(pneu).astype(np.float32), labels(pneu)

fig, axes = plt.subplots(1, len(names), figsize=(3.2 * len(names), 3.4))
for ax, cls in zip(axes, range(len(names))):
    mean_img = X[y == cls].mean(axis=0)
    ax.imshow(mean_img, cmap="gray")
    ax.set_title(f"{names[cls]}  (n={(y == cls).sum()})")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("PneumoniaMNIST — mean training image per class", y=1.02)
fig.tight_layout()
out = FIG_DIR / "pneumonia_mean_per_class.png"
fig.savefig(out)
print("saved", out.name)
plt.show()

In [ ]:
# Per-channel statistics: BloodMNIST is RGB; the other two are single-channel.
rows = []
for name, splits in datasets.items():
    X = imgs(splits["train"]).astype(np.float32) / 255.0
    if X.ndim == 3:
        rows.append({"dataset": name, "channel": "gray",
                     "mean": float(X.mean()), "std": float(X.std())})
    else:
        for ci, ch in enumerate("RGB"):
            rows.append({"dataset": name, "channel": ch,
                         "mean": float(X[..., ci].mean()),
                         "std":  float(X[..., ci].std())})
channel_stats = pd.DataFrame(rows)
channel_stats

## 6. First OOD-flavored visualization

One figure, three rows of samples — one per dataset. The visual gap between rows is the **intuition for far-OOD**: BloodMNIST (RGB microscopy) is obviously different from a chest X-ray; OrganAMNIST (grayscale CT) is closer in style and therefore harder to flag as OOD.

In [ ]:
n_per_row = 6
fig, axes = plt.subplots(3, n_per_row, figsize=(n_per_row * 1.5, 4.8))
for r, (name, splits) in enumerate(datasets.items()):
    X = imgs(splits["train"])
    is_rgb = (X.ndim == 4)
    pick = RNG.choice(len(X), size=n_per_row, replace=False)
    for c in range(n_per_row):
        ax = axes[r, c]
        ax.imshow(X[pick[c]] if is_rgb else X[pick[c]],
                  cmap=None if is_rgb else "gray")
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        if c == 0:
            ax.set_ylabel(name, rotation=0, ha="right", va="center",
                          fontsize=11, labelpad=10,
                          color=DATASET_COLORS[name])
fig.suptitle("Visual gap between candidate ID and OOD datasets", fontsize=12)
fig.tight_layout()
out = FIG_DIR / "ood_visual_gap.png"
fig.savefig(out)
print("saved", out.name)
plt.show()

## 7. Takeaways for the project

- **PneumoniaMNIST is imbalanced** in favor of the *Pneumonia* class (see Section 3). Plain accuracy will be misleading — use **AUROC** and **balanced accuracy**, and consider a **class-weighted loss** when training the BNN.
- **Sizes are small** (~5k train for Pneumonia, tens of thousands for the others) at 28×28 — full training fits comfortably on a single GPU, leaving budget for many BNN posterior samples / Laplace experiments.
- **BloodMNIST is the easy far-OOD baseline**: RGB microscopy, completely different pixel-intensity distribution (Section 5 overlay). If our OOD detector cannot separate it from PneumoniaMNIST, something is wrong.
- **OrganAMNIST is the harder near-OOD case**: also grayscale medical imaging with similar intensity statistics. This is the realistic test of whether BNN uncertainty actually reflects distributional shift.
- **Channel mismatch matters**: ID is single-channel, BloodMNIST is RGB. Whatever input pipeline we build in `src/` must convert consistently (e.g. RGB→gray or replicate gray→3 channels) so OOD evaluation is apples-to-apples.
- **Mean-image sanity check** on PneumoniaMNIST (Section 5) shows no obvious systematic artifact between classes — no immediate red flag for label leakage from positioning/cropping.